In [ ]:
# ── 第一格：环境准备（D02 封装之后，每本 notebook 开头都是这十几行）──
%load_ext autoreload
%autoreload 2
# 改了 src/ 里的 .py，notebook 不用重启就能用上新代码
from signal_mysql import database as mysql
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from signal_insight import config, geo, privacy, quality, viz
from signal_insight.viz import MAIN, MAIN_LIGHT, MAIN_DARK, MAIN_MID, GRAY, GRAY_DARK, RED, AMBER, INK, MUTED, FAINT, AREA_COLOR, SEG_COLOR  # 配色常量
viz.apply_theme()                           # 字体、配色、坐标轴样式：一行
con = mysql.connect(config.DB_SCHEMA)       # 路径从包里来：notebook 在哪个文件夹打开都找得到库
DAY = config.ANALYSIS_DAY                   # 分析日期（口径常量）
NB = "D10_报告与答辩"
SOURCE = f"{config.SOURCE_NOTE}；脚本：{NB}"    # 每张图左下角的来源行
pd.set_option("display.max_columns", 20)

In [ ]:
from signal_mysql import database as mysql
# 【跟丢了才运行】前几天没做完的同学：把 False 改成 True 再运行，从检查点库复制标准表
CATCH_UP = False
if CATCH_UP:
    with mysql.connect(config.CHECKPOINT_SCHEMA) as ck:
        for t in ["dwd_signal", "dim_cell", "dim_hotspot", "dws_stay", "dws_trip", "dws_night", "dws_user_day", "dws_user_hour"]:
            mysql.write_frame(mysql.read_frame(f"SELECT * FROM {t}", ck), t, con, if_exists="replace", index=False)
            print("已恢复", t)

In [ ]:
from signal_mysql import database as mysql
settings = config.load_settings()
hot = mysql.read_frame("SELECT * FROM dim_hotspot", con).iloc[0]                  # 西塔中心（D4）
DAY_START = pd.Timestamp(DAY)
REPORT_DIR = config.REPORT_DIR
REPORT_DIR.mkdir(exist_ok=True)
DISTRICTS = geo.rings_from_points(mysql.read_frame("SELECT * FROM ods_district_boundary", con), "district")
stay = mysql.read_frame("SELECT * FROM dws_stay", con, parse_dates=["start_time", "end_time"])
night = mysql.read_frame("SELECT * FROM dws_night", con)
day = mysql.read_frame("SELECT * FROM dws_user_day", con)
poi = mysql.read_frame("SELECT * FROM ods_poi_purpose", con)
assert "segment_v2" in day.columns, "dws_user_day 里没有 segment_v2：运行 python -m signal_insight.pipeline，或运行上面的“跟丢了才运行”"
print(f"用户 {len(day):,}；傍晚访客 {int(day['is_evening_visitor'].sum()):,}；停留 {len(stay):,}；过夜地 {len(night):,}")
from signal_mysql import database as mysql
ev = day[day["is_evening_visitor"] == 1]
uh = mysql.read_frame("SELECT hour, dist_xita_m FROM dws_user_hour", con)
x19 = int(((uh["hour"] == 19) & (uh["dist_xita_m"] <= 500)).sum())
x23 = int(((uh["hour"] == 23) & (uh["dist_xita_m"] <= 500)).sum())
n19 = int((uh["hour"] == 19).sum())
visitors_500_30 = int(stay.loc[stay["dist_xita_m"] <= 500, "imsi"].nunique())
outsider = int((ev["segment_v2"] == "外地来客").sum())
after23 = ev["xita_after_23"].mean()
tables = {r[0] for r in con.execute('SELECT table_name AS name FROM information_schema.tables WHERE table_schema=DATABASE()')}
# 这个 if/else 是给"D8 选讲部分没跑"的班级兜底：库里没有这张表时，
# 退回用 D8 的对账数字（1,836 / 5,282）。
# 讲台提示：企业里的报告脚本也该这么写——依赖的上游表缺失时要有明确的降级行为，
# 而不是直接抛异常让整份报告生成失败。
if "ads_caliber_visitors" in tables:                                    # D8 需求卡 8.5 的结果表
    cal = mysql.read_frame("SELECT n_users FROM ads_caliber_visitors", con)["n_users"]
    lo, hi = int(cal.min()), int(cal.max())
else:
    lo, hi = 1_836, 5_282                                               # 库里没有这张表时，用 D8 的对账数字
# 这一格在做什么：把前十天的发现组装成一份 8 页的汇报大纲。
#
# 注意下面 pages 的顺序：**结论在第 1 页**，然后才是数据与口径、三条发现、局限、建议。
# 学生的本能是按时间顺序讲"我们先做了什么、再做了什么"——那是日志，不是汇报。
# 客户只有 10 分钟，第一页没讲出结论，后面讲什么都来不及了。
#
# 还要注意第 1 条结论写的是**样本范围**："结论只对这群人成立"。
# 把限制放在结论的第一条，而不是藏在最后的附录里——这是 D4 改写需求时就定下的态度。
#
# 四条结论里的数字全部由代码从库里算出来（x19、x23、outsider 等变量），
# 和图表标题是同一个原则：手敲的数字早晚会和表对不上。
conclusions = [
    f"这份数据是 10 月 3 日傍晚到访西塔的 {len(day):,} 名移动用户及其全天信令，结论只对这群人成立。",
    f"19 点西塔 500 米内有 {x19:,} 人，占当时可定位人数的 {x19 / n19:.0%}；23 点仍有 {x23:,} 人，{after23:.0%} 的傍晚访客在数据截止时还没走。",
    f"傍晚访客里 {outsider / len(ev):.0%}（{outsider} 人）是当天从外地进城的；只按过夜地判断，只能找到 1%。",
    f"“到访人数”随口径在 {lo:,} 到 {hi:,} 人之间；本报告统一用“500 米内停留 30 分钟以上”：{visitors_500_30:,} 人。",
]
pages = [
    ("结论", conclusions),
    ("数据与口径", ["样本：中国移动用户，按 10 月 3 日傍晚到访西塔圈选；只有一天，国庆假期",
                  "在场人数：每人每小时一个位置（dws_user_hour）；到访：500 米内停留 30 分钟以上（dim_metric）"]),
    ("发现 1：西塔的晚高峰", ["图 D9_01：19 点达到高峰，23 点仍有上千人"]),
    ("发现 2：外地来客被低估", ["图 D8_04：修正前后的客群构成"]),
    ("发现 3：口径决定数字", ["图 D8_02：六种算法，相差近 3 倍"]),
    ("局限", ["样本圈选、只有一天、只有一家运营商、基站级精度、23:59 截止、过夜地只识别出约六成"]),
    ("建议", ["每条写清谁、做什么、对应哪条发现"]),
    ("附录", ["数据质量主要问题（作业 1）；指标口径表 dim_metric"]),
]
outline = "# 汇报大纲（示范）：国庆夜间的西塔\n\n"
for i, (head, items) in enumerate(pages, 1):
    outline += f"## 第 {i} 页　{head}\n" + "".join(f"- {t}\n" for t in items) + "\n"
(REPORT_DIR / "汇报大纲_示范.md").write_text(outline, encoding="utf-8")
print(outline)

In [ ]:
kx, ky = 111.320 * np.cos(np.radians(hot["lat"])), 110.574           # 以西塔为原点的平面坐标（公里）
to_xy = lambda lng, lat: np.column_stack([(np.array(lng) - hot["lng"]) * kx, (np.array(lat) - hot["lat"]) * ky])
def nearest_km(points, refs, chunk=2000):
    """每个点到参照点集中最近一点的距离（公里），分块计算，省内存。"""
    points, refs = np.array(points, dtype=np.float32), np.array(refs, dtype=np.float32)
    out = np.empty(len(points), dtype=np.float32)
    for s in range(0, len(points), chunk):
        a = points[s:s + chunk]
        d2 = (a[:, :1] - refs[None, :, 0]) ** 2 + (a[:, 1:] - refs[None, :, 1]) ** 2
        out[s:s + chunk] = np.sqrt(d2.min(axis=1))
    return out
st = stay.copy()
sxy = to_xy(st["lng"], st["lat"])
for t, col in [("购物", "shop_km"), ("旅游", "scenic_km")]:
    p = poi[poi["type"] == t]
    st[col] = nearest_km(sxy, to_xy(p["lng"], p["lat"]))
st = st.join(night.set_index("imsi")[["night_lng", "night_lat"]], on="imsi")
st["home_m"] = geo.haversine(st["lat"], st["lng"], st["night_lat"], st["night_lng"])

In [ ]:
ev_ids = day.loc[day["is_evening_visitor"] == 1, "imsi"]
n_ev = len(ev_ids)
s = st[st["imsi"].isin(ev_ids)]
ex = (s[(s["dist_xita_m"] <= 500) & (s["end_time"] >= DAY_START + pd.Timedelta(hours=17))]
      .groupby("imsi")["start_time"].min().rename("ex"))                   # 傍晚那次西塔停留的开始时刻
s = s.join(ex, on="imsi")
b = s[(s["end_time"] <= s["ex"]) & (s["dist_xita_m"] > 500)]                 # 在那之前、西塔以外的停留
home = b["home_m"] <= 500
in_aoi = b["aoi"].isin(["太原街商圈", "中街商圈"])
shop = (b["shop_km"] <= 0.3) & ~in_aoi & ~home
flags = pd.DataFrame({"imsi": b["imsi"], "过夜地（从住处出发）": home,
                      "太原街商圈": b["aoi"] == "太原街商圈", "中街商圈": b["aoi"] == "中街商圈",
                      "其他购物点附近": shop, "景点附近": (b["scenic_km"] <= 0.3) & ~in_aoi & ~home & ~shop})
flags["其他地点"] = ~flags.drop(columns="imsi").any(axis=1)
per_user = flags.groupby("imsi").any()
cnt = per_user.sum()
cnt["当天没有更早的停留"] = n_ev - len(per_user)
share = (cnt / n_ev).sort_values()
gaps = {}
for name in ["太原街商圈", "中街商圈"]:
    last = b[b["aoi"] == name].groupby("imsi")["end_time"].max()
    g = (ex.reindex(last.index) - last).dt.total_seconds() / 60
    gaps[name] = (len(last), last.dt.hour.median(), g.median())
after_ty = s[(s["aoi"] == "太原街商圈") & (s["start_time"] >= s["ex"])]["imsi"].nunique()

In [ ]:
for k, (n, hh, m) in gaps.items():
    print(f"{k}：{n} 人（{n / n_ev:.1%}），离开时刻中位 {hh:.0f} 点，离开后中位 {m:.0f} 分钟到西塔")
print(f"离开西塔之后又去太原街的：{after_ty} 人")
# 对账
assert n_ev == 3_310 and cnt["太原街商圈"] == 498 and cnt["中街商圈"] == 370
print("✅ 对账通过：傍晚访客 3,310 人；来西塔前在太原街停留的 498 人，在中街的 370 人")

In [ ]:
# “之后”用和对账数字（离开西塔后去太原街 231 人）相同的定义：西塔傍晚停留开始之后开始的、西塔以外的停留；
# 太原街、中街的停留即使落在西塔 500 米内也算（两个商圈的边界和西塔圈有少量重叠）
a_ = s[(s["start_time"] >= s["ex"]) & ((s["dist_xita_m"] > 500) | s["aoi"].isin(["太原街商圈", "中街商圈"]))]
home_a = a_["home_m"] <= 500
in_aoi_a = a_["aoi"].isin(["太原街商圈", "中街商圈"])
shop_a = (a_["shop_km"] <= 0.3) & ~in_aoi_a & ~home_a
fa = pd.DataFrame({"imsi": a_["imsi"], "过夜地": home_a, "太原街商圈": a_["aoi"] == "太原街商圈",
                   "中街商圈": a_["aoi"] == "中街商圈", "其他购物点附近": shop_a,
                   "景点附近": (a_["scenic_km"] <= 0.3) & ~in_aoi_a & ~home_a & ~shop_a})
fa["其他地点"] = ~fa.drop(columns="imsi").any(axis=1)
after = fa.groupby("imsi").any().sum()
after["没有其他停留"] = n_ev - fa["imsi"].nunique()
before = cnt.rename({"过夜地（从住处出发）": "过夜地", "当天没有更早的停留": "没有其他停留"})
assert after["太原街商圈"] == after_ty                                   # 和上一格打印的对账数字一致
# ── 出图 D10_01：对向条形图（蝴蝶图）─────────────────────────────────
# 为什么用左右对向而不是分组条形：要比较的是同一批人"来之前"和"之后"的差别，
# 共用一条中轴、往两边长，差异一眼就看出来；分组条形还要对照图例分辨哪根是哪个。
# 为什么**不用桑基图**（D8_04 用了）：这里一个人可以同时属于多类
# （既去过太原街也去过景点），比例加起来超过 100%，桑基的带宽就没有意义了。
# 副标题里特意写明了这一点。
#   **同一批人重新分配 → 桑基；一人可属多类 → 对向条形。**
# 规范落在哪几行：
#   · 只有太原街、中街两行上主色——这张图要讲的就是它俩
#   · 左右两侧写 "← 来西塔之前" "离开西塔之后 →"——对向图不标方向就读不懂
#   · axvline(0) 画出中轴——对向条形必须有一条明确的基线
# 标题里的数字：before["太原街商圈"] / n_ev 和 after["太原街商圈"] / n_ev。
rows_ = ["没有其他停留", "其他地点", "景点附近", "其他购物点附近", "过夜地", "中街商圈", "太原街商圈"]
fig, ax = plt.subplots(figsize=(10, 5.6), layout="constrained")
for i, kk in enumerate(rows_):
    hi = kk in ("太原街商圈", "中街商圈")
    for val, sign, c_ in [(before[kk] / n_ev, -1, MAIN if hi else GRAY), (after[kk] / n_ev, 1, MAIN_MID if hi else "#DDE1E7")]:
        ax.barh(i, sign * val, color=c_, height=0.6)
        ax.text(sign * (val + 0.008), i, f"{val:.1%}", ha="right" if sign < 0 else "left", va="center", fontsize=9.5,
                color=INK, fontweight="bold" if hi else "normal")
ax.axvline(0, color=INK, lw=1)
ax.set_yticks(range(len(rows_)), rows_, fontsize=10, color=INK)
ax.set_xlim(-0.62, 0.62); ax.set_xticks([]); ax.grid(False); ax.set_ylim(-0.6, len(rows_) + 0.2)
ax.text(-0.02, len(rows_) - 0.3, "← 来西塔之前", ha="right", fontsize=10, color=MAIN, fontweight="bold")
ax.text(0.02, len(rows_) - 0.3, "离开西塔之后 →", ha="left", fontsize=10, color=MAIN_DARK, fontweight="bold")
title = (f"{before['太原街商圈'] / n_ev:.0%} 的傍晚访客先在太原街停留再来西塔，"
         f"之后再去太原街的只有 {after['太原街商圈'] / n_ev:.0%}")
viz.finish(fig, title,
           f"{n_ev:,} 个傍晚访客当天在西塔以外的停留，按地点类型分；一人可属多类，比例不能相加（所以不用桑基图）；\n“之前”= 在傍晚到达西塔前结束，“之后”= 在那之后开始；购物点、景点 = 300 米内有该类 POI；口径：dws_stay、dws_night、ods_poi_purpose",
           SOURCE, "D10_01_商圈联动.png")

In [ ]:
# ── 出图 D10_02：热力地图 + 同心圈层 + ECDF ───────────────────────────
# 为什么把圈层画成**椭圆**：50%、70%、90% 分位对应的是以西塔为心的等距离圆，
# 但经纬度坐标下 1 度经度 ≈ 0.75 度纬度，正圆画出来是扁的。
# 所以长短轴分别除以 kx（经度方向每度多少公里）和 111.32（纬度方向），画出来才是真的圆。
# 这是"地图长宽比校正"在非地图元素上的同一条规范。
# 为什么右边还要配 ECDF：地图回答"在哪"，ECDF 回答"多远"。
# 圈层的 50/70/90 三个数在两张图上**同时出现**，读者能把两者对上。
# 规范落在哪几行：
#   · mincnt=5 —— 小样本抑制
#   · norm=LogNorm(5, 200) —— 对数色阶，跨数量级
#   · 副标题写"外地来客几乎都识别不出过夜地、不在其中，真实辐射范围比图上大"——
#     **这是口径限制里最关键的一句**：这张图系统性低估了远距离客源，必须写出来
# 标题里的数字：q_[50]、q_[70]、q_[90]，即距离的三个分位数。
from matplotlib.colors import LinearSegmentedColormap, LogNorm
from matplotlib.patches import Ellipse
d = ev["night_dist_xita_km"].dropna()
q_ = {kk: d.quantile(kk / 100) for kk in (50, 70, 90)}
r50 = q_[50]
print(f"1 公里内 {(d <= 1).mean():.0%}，20 公里外 {(d > 20).mean():.1%}")
pts = night[night["imsi"].isin(ev.loc[ev["night_dist_xita_km"].notna(), "imsi"])]
x0, x1 = hot["lng"] - 0.165, hot["lng"] + 0.165; y0, y1 = hot["lat"] - 0.115, hot["lat"] + 0.115    # 西塔周边约 12 公里
BLUES = LinearSegmentedColormap.from_list("blues", ["#EAF2F9", MAIN_MID, MAIN, MAIN_DARK])    # 单色渐变，不用彩虹色阶
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 5.6), layout="constrained", width_ratios=[1.35, 1])
viz.maps.draw_districts(a1, DISTRICTS, edge="#C4CAD3")
a1.hexbin(pts["night_lng"], pts["night_lat"], gridsize=(40, 20), extent=(x0, x1, y0, y1), mincnt=5, cmap=BLUES,
          norm=LogNorm(5, 200), lw=0.2, edgecolors="white", zorder=2, alpha=0.85)
kx = 111.32 * np.cos(np.radians(41.8))
for kk, name, ls, ang in [(50, "核心圈", "-", 140), (70, "次级圈", "--", 38), (90, "边缘圈", ":", -38)]:
    a1.add_patch(Ellipse((hot["lng"], hot["lat"]), 2 * q_[kk] / kx, 2 * q_[kk] / 111.32, fill=False, ec=INK, lw=1.4, ls=ls, zorder=5))
    a1.text(hot["lng"] + q_[kk] / kx * np.cos(np.radians(ang)), hot["lat"] + q_[kk] / 111.32 * np.sin(np.radians(ang)),
            f"{name}  {q_[kk]:.1f} 公里 · {kk}%", fontsize=8.8, color=INK, fontweight="bold", zorder=6,
            ha="right" if ang > 90 else "left", va="bottom" if ang > 0 else "top",
            bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.85))
a1.plot(hot["lng"], hot["lat"], marker="*", ms=12, color=INK, zorder=7)
a1.set_xlim(x0, x1); a1.set_ylim(y0, y1)
viz.maps.set_map_aspect(a1)           # 长宽比校正 + 去掉刻度、网格、边框
viz.maps.scale_bar(a1, 2); viz.maps.north_arrow(a1)
a2.ecdf(d, color=MAIN, lw=2)
for kk in (50, 70, 90):
    a2.plot([q_[kk], q_[kk]], [0, kk / 100], color=MUTED, lw=0.9, ls="--"); a2.plot(q_[kk], kk / 100, "o", color=MAIN, ms=5)
    a2.text(q_[kk] + 0.5, kk / 100 - 0.06, f"{kk}%：{q_[kk]:.1f} 公里", fontsize=9, color=INK)
a2.set_xlim(0, 20); a2.set_ylim(0, 1)
a2.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
a2.set_xlabel("过夜地到西塔的直线距离（公里）"); a2.grid(axis="both")
a2.set_title("累计分布", loc="left", fontsize=10.5, color=INK, fontweight="bold")
title = f"一半的傍晚访客住在西塔 {q_[50]:.1f} 公里内，七成 {q_[70]:.1f} 公里内，九成 {q_[90]:.1f} 公里内"
viz.finish(fig, title,
           f"能识别过夜地的 {len(d):,} 个傍晚访客（占 {len(d) / len(ev):.0%}）；外地来客几乎都识别不出过夜地、不在其中，真实辐射范围比图上大；六边形少于 5 人不显示",
           SOURCE, "D10_02_辐射圈层.png")
# 对账
assert len(d) == 2_067 and round(r50, 2) == 2.66
print(f"✅ 对账通过：{len(d):,} 人，一半住在 {r50:.2f} 公里内")

In [ ]:
# 这一格在做什么：交付前自查——把前十天该有的表和行数列出来，逐个核对。
#
# 为什么要写成一张表而不是靠记忆：十天下来库里有十几张表，
# 哪一步重跑过、哪一步漏了，光靠回忆一定会出错。
# D15 会把这件事升级成 tools/preflight.py（11 项检查，一条命令跑完），这里是它的雏形。
#
# 讲台提示：企业里交付前都有这么一张"冒烟检查表"。
# 让学生养成"交之前先自己跑一遍清单"的习惯，比教任何技术都管用。
need = {"dwd_signal": 515_931, "dim_cell": 49_948, "dim_hotspot": 1, "dws_stay": 19_015, "dws_trip": 11_055,
        "dws_night": 3_607, "dws_user_day": 5_644, "dws_user_hour": 102_859}
have = {r[0] for r in con.execute('SELECT table_name AS name FROM information_schema.tables WHERE table_schema=DATABASE()')}
rows = []
for t, n in need.items():
    got = con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0] if t in have else None
    rows.append({"表": t, "行数": got, "参考": n, "一致": got == n})
check = pd.DataFrame(rows)
check

In [ ]:
# 对账
assert check["一致"].all(), check[~check["一致"]]
print("✅ 对账通过：8 张主要表的行数全部和参考一致")

In [ ]:
from signal_mysql import database as mysql
DOCS = config.PROJECT_ROOT / "docs"
if "dim_metric" in have:
    from signal_insight.dims import METRIC_LABELS           # 库里的列名是英文；给人看的文档用中文表头
    mysql.read_frame("SELECT * FROM dim_metric", con).rename(columns=METRIC_LABELS).to_csv(DOCS / "指标口径表.csv", index=False, encoding="utf-8-sig")
    print("已导出 docs/指标口径表.csv（.gitignore 放行了 docs/ 下的 CSV，它会进仓库）")
# README 自查：从 D01 起一直在补的 README，交付前看看该有的几节都在不在
readme = (config.PROJECT_ROOT / "README.md").read_text(encoding="utf-8")
need_parts = {"目录说明": "目录", "怎么安装": "pip install -e", "怎么跑流水线": "signal_insight.pipeline", "怎么跑测试": "pytest"}
readme_check = pd.DataFrame([{"README 应有": k, "找到": v in readme} for k, v in need_parts.items()])
print("出图：", len(list(config.IMAGE_DIR.glob("*.png"))), "张")
readme_check

In [ ]:
import ast
import json
import re
import subprocess
SRC = config.PROJECT_ROOT / "src" / "signal_insight"
COMPUTE = ["dwd.py", "dims.py", "privacy.py"] + [f"dws/{p.name}" for p in (SRC / "dws").glob("*.py")]
BUSINESS = {500, 30, 17, 23, 1.5}
def tree(p):
    return ast.parse(p.read_text(encoding="utf-8"))
def imports(p):
    out = set()
    for n in ast.walk(tree(p)):
        if isinstance(n, ast.Import):
            out |= {a.name for a in n.names}
        elif isinstance(n, ast.ImportFrom):
            out.add(n.module or "")
    return out
def calls(p):
    return {n.func.attr if isinstance(n.func, ast.Attribute) else getattr(n.func, "id", "")
            for n in ast.walk(tree(p)) if isinstance(n, ast.Call)}
rows = []
bad1 = [str(p.relative_to(SRC)) for p in SRC.rglob("*.py") if any("notebook" in m for m in imports(p))]
rows.append(("1 src 不 import notebooks", "、".join(bad1) or "没有", not bad1))
bad2 = [m for m in COMPUTE if any("sqlite" in x for x in imports(SRC / m)) or calls(SRC / m) & {"read_sql", "to_sql", "connect"}]
rows.append(("2 计算模块不碰库", "、".join(bad2) or "没有", not bad2))
# 只查计算模块和流水线：viz 里的 linespacing=1.5 是排版参数，不是业务数字（自动检查要限定范围，否则误报会让人不再相信它）
bad3 = [f"{m}:{n.lineno} 出现 {n.value}" for m in COMPUTE + ["pipeline.py"] for n in ast.walk(tree(SRC / m))
        if isinstance(n, ast.Constant) and type(n.value) in (int, float) and n.value in BUSINESS]
rows.append(("3 代码里没有业务数字", "；".join(bad3) or "没有", not bad3))
bad4 = []
for nbp in sorted((config.PROJECT_ROOT / "notebooks").glob("D*.ipynb")):
    codes = ["".join(c["source"]) for c in json.load(open(nbp, encoding="utf-8"))["cells"] if c["cell_type"] == "code"]
    # 找标着“第一格”的那一格（D02 的第一格在四张封装卡之后，前面还有一格验证可编辑安装）
    src = next((s for s in codes if s.startswith("# ── 第一格")), "")
    if "viz.apply_theme()" not in src or "def " in src or len(src.splitlines()) > 25:
        bad4.append(nbp.name[:3])
rows.append(("4 notebook 第一格只剩 import", "、".join(bad4) or "全部符合", not bad4))
try:
    log = subprocess.run(["git", "log", "--format=%s"], cwd=config.PROJECT_ROOT, capture_output=True, text=True, encoding="utf-8", check=True).stdout.splitlines()
    pat = re.compile(r"^(feat|refactor|fix|test|docs|chore|perf|analysis): \S")
    # D01–D06 的旧提交在 2026-10-21 复查时决定保持现状：已经推送，重写历史的代价大于收益。
    # 这里把它们列成已知欠账，不阻塞评审；D07 起的新提交必须合规范。
    LEGACY = {"修复已知问题", "代码格一改为调用quality中的profile，增加复用性",
              "refactor:将profile收进quality.py,D001第一格改为调用", "notebooks:D001代码优化", "-a",
              "fix:文件名称更改signal-insight-->signal_insight", "docs:发现清单F1-F4",
              "anaysis:D001 读ODS、画像、按天按小时统计"}
    bad6 = [m for m in log if not pat.match(m) and m not in LEGACY]
    rows.append(("6 提交说明合规", f"{len(log)} 个提交，{len(bad6)} 个不合规" + (f"：{bad6[:3]}" if bad6 else ""), not bad6))
except (FileNotFoundError, subprocess.CalledProcessError):
    rows.append(("6 提交说明合规", "这里不是 Git 仓库，跳过", True))
review = pd.DataFrame(rows, columns=["检查项", "结果", "通过"])
review

In [ ]:
# 对账：评审脚本在参考仓库上全部通过；你的仓库不通过的项，一处一个提交改掉
assert review["通过"].all(), review[~review["通过"]]
print("✅ 中期代码评审：能自动查的 5 项全部通过（第 5、7 项人工查：对照 docs/工程规范.md；python -m pytest）")

In [ ]:
from signal_mysql import database as mysql
try:
    from sklearn.cluster import KMeans
    from sklearn.metrics import silhouette_score
except ImportError:
    KMeans = None
    print("没有安装 scikit-learn，跳过本卡：pip install scikit-learn")
if KMeans is not None:
    users = day["imsi"]
    u = mysql.read_frame("SELECT imsi, hour, lng, lat, dist_xita_m FROM dws_user_hour", con)
    u = u.join(night.set_index("imsi")[["night_lng", "night_lat"]], on="imsi")
    at_home = geo.haversine(u["lat"], u["lng"], u["night_lat"], u["night_lng"]) <= 500
    u["state"] = np.select([u["dist_xita_m"] <= 500, at_home], ["X", "H"], default="O")
    mats = [(u[u["state"] == s_].assign(v=1.0).pivot_table(index="imsi", columns="hour", values="v", aggfunc="max")
             .reindex(index=users, columns=range(24)).fillna(0.0)) for s_ in ["X", "H", "O"]]
    X = np.hstack([m.to_numpy() for m in mats])
    # random_state 固定住：聚类每次跑结果都一样，否则对账数字和图都对不上。
    # 这和 D5 的"幂等"是同一个要求——有随机性的步骤必须固定种子。
    km = KMeans(n_clusters=5, n_init=20, random_state=42).fit(X)
    # 轮廓系数：衡量"类分得开不开"，越接近 1 越好。这里只有 0.2 左右，
    # 说明类与类的边界是模糊的。**这个数必须报出来**（图的副标题里也写了），
    # 不报的话读者会以为这 5 类是泾渭分明的。
    sil = silhouette_score(X, km.labels_, sample_size=3000, random_state=0)
    # 给每一类起名字：看它的中心在哪些小时、哪种状态上最高
    C = km.cluster_centers_
    Cx, Ch, Co = C[:, :24], C[:, 24:48], C[:, 48:]
    Cu = 1 - Cx - Ch - Co
    names, remaining = {}, list(range(5))
    for score, name in [(lambda k: Cx[k, :17].mean(), "西塔常驻"), (lambda k: Cu[k, :12].mean(), "午后才出现"),
                        (lambda k: Ch[k, 10:16].mean(), "宅到下午"), (lambda k: Ch[k, :6].mean(), "早出晚归")]:
        k = max(remaining, key=score)
        names[k] = name
        remaining.remove(k)
    names[remaining[0]] = "白天在外"
    lab = pd.Series(km.labels_, index=users.to_numpy(), name="作息类型").map(names)
    print(f"轮廓系数 {sil:.2f}（越接近 1 类分得越开；这里偏低，类的边界是模糊的）")
    print(lab.value_counts().to_string())

In [ ]:
if KMeans is not None:
    # ── 出图 D10_03：小多图（small multiples）───────────────────────────
    # 为什么拆成 5 张小图而不是画在一张图上：5 类 × 4 种状态 = 20 条线，
    # 挤在一张图里谁也看不清。拆开之后每张图只有 4 层，而且**共用纵轴（sharey=True）**，
    # 横着扫一眼就能比较五类人的作息差异。
    # 这就是"小多图"：宁可多几张图小图，不要一张挤满的大图。
    # 规范落在哪几行：
    #  · sharey=True + set_ylim(0, 100) —— 不共轴、不固定范围的话，五张图的高度没法比
    #  · 每张子图标题带上"占比（人数）" —— 读者要知道这一类有多少人
    #  · 堆叠的四层互斥且合计 100%（在西塔 / 在别处 / 在过夜地 / 不可定位）—— 符合堆叠的前提
    #  · 副标题写轮廓系数并注明"只作探索" —— 聚类结果的可信度必须交代
    # 标题里没有数字：这张图的结论是定性的（"能互相印证"）。
    # 规范要求标题带数字，定性结论的图是少数例外，可信度放在副标题里交代。
    order = ["早出晚归", "宅到下午", "白天在外", "午后才出现", "西塔常驻"]
    seg = day.set_index("imsi")["segment_v2"]
    fig, axes = plt.subplots(1, 5, figsize=(12, 4.8), sharey=True)
    for ax, name in zip(axes, order):
        idx = (lab == name).to_numpy()
        xs, hs, os_ = (m.to_numpy()[idx].mean(axis=0) for m in mats)
        un = 1 - xs - hs - os_
        ax.stackplot(range(24), xs * 100, os_ * 100, hs * 100, un * 100,
                     colors=[MAIN, "#C9CED6", "#6B7684", "#F1F3F5"], edgecolor="none")
        n = int(idx.sum())
        ax.set_title(f"{name}\n{n / len(lab):.0%}（{n:,} 人）", loc="left", fontsize=11, color=INK)
        ax.set_xlim(0, 23)
        ax.set_ylim(0, 100)
        ax.set_xticks([0, 12, 23], ["0 点", "12 点", "23 点"])
        ax.grid(False)
    axes[0].set_yticks([0, 50, 100], ["0", "50%", "100%"])
    title = "KMeans 从 24 小时轨迹里分出 5 类作息，和规则标签能互相印证"
    viz.finish(fig, title,
               f"每人每小时的状态展开成 72 维特征，k = 5；蓝 = 在西塔，浅灰 = 在别处，深灰 = 在过夜地，最浅 = 不可定位；\n"
               f"轮廓系数 {sil:.2f}，类与类的边界模糊，只作探索；口径：dws_user_hour、dws_night",
               SOURCE, "D10_03_作息聚类.png")
    cross = pd.crosstab(lab, seg.reindex(lab.index).rename("规则标签"), normalize="index").reindex(order).round(3)
    cross

In [ ]:
try:
    from pptx import Presentation
    from pptx.util import Cm, Pt
except ImportError:
    Presentation = None
    print("没有安装 python-pptx，跳过本卡：pip install python-pptx")
if Presentation is not None:
    prs = Presentation()
    prs.slide_width, prs.slide_height = Cm(33.867), Cm(19.05)           # 16:9
    def new_slide(title):
        sl = prs.slides.add_slide(prs.slide_layouts[5])                  # 只有标题的版式
        t = sl.shapes.title
        t.left, t.top, t.width, t.height = Cm(2), Cm(0.8), prs.slide_width - Cm(4), Cm(2.6)
        t.text = title
        t.text_frame.paragraphs[0].font.size = Pt(28)
        return sl
    s0 = new_slide("国庆夜间的西塔：结论")
    box = s0.shapes.add_textbox(Cm(2), Cm(4.5), Cm(29.8), Cm(12)).text_frame
    box.word_wrap = True
    for i, c in enumerate(conclusions):
        p = box.paragraphs[0] if i == 0 else box.add_paragraph()
        p.text, p.font.size, p.space_after = f"{i + 1}. {c}", Pt(20), Pt(12)
    figs = [("西塔的晚高峰：19 点达到高峰，23 点仍有上千人", "D9_01_三商圈的一天.png"),
            ("外地来客被低估：修正后占傍晚访客的 14%", "D8_04_客群口径修正.png"),
            ("口径决定数字：同一个“到访人数”相差近 3 倍", "D8_02_到访人数口径对比.png")]
    for head, fn in figs:
        path = config.IMAGE_DIR / fn
        if not path.exists():
            print("缺图，跳过：", fn)
            continue
        sl = new_slide(head)
        from PIL import Image
        w, h = Image.open(path).size
        box_w, box_h = prs.slide_width - Cm(4), Cm(14.6)              # 图放在标题下面的这个框里，按比例缩放
        if box_w * h / w <= box_h:
            sl.shapes.add_picture(str(path), Cm(2), Cm(3.8), width=box_w)
        else:
            width = int(box_h * w / h)
            sl.shapes.add_picture(str(path), int((prs.slide_width - width) / 2), Cm(3.8), height=box_h)
    out = REPORT_DIR / "汇报初稿_示范.pptx"
    prs.save(out)
    print(f"已生成 {out.name}：{len(prs.slides)} 页")

In [ ]:
con.close()